# East Bay corridor activity: full monthly extraction, 2019–2025

This version implements **monthly stop batches, one provider per half-year, and Bay Area HOME only**. Run from top to bottom in the same Cuebiq environment used for the pilots. Check the paths/schema settings in section 1. Existing v1 checkpoints cannot be reused because provider and HOME definitions changed.

## Output and methods contract
- All 11 Oakland BID corridors; January 2019–December 2025. Geohash9 cover ≥0.8, equal cell weights. Shared cells count in each intersecting corridor, so corridor totals cannot be summed into distinct East Bay visitors.
- Raw stop counts and device-day visits remain distinct. One ID/corridor/local date is one visit. Full dwell must be positive and ≤480 minutes. Duplicate event-partition copies are removed. No TAC gap-session definition.
- Five overlapping periods: all; weekdays; weekends; weekday 09:00–16:59; evening 17:00–22:59 every day. Do not sum periods.
- Provider screening uses the 10th of every month, all candidate providers and all corridors together. For each half-year, select the provider with the largest summed sampled unique-device count among candidates present on all six dates. This estimates coverage, not an exhaustive full-month ranking. Selection does not prefer a provider because its later outcome trend looks favourable. The schedule is exported; overrides are explicit and fingerprinted.
- One provider per half-year avoids cross-provider pooling of possibly duplicated phones. Same-provider IDs still measure devices, not verified people. Provider changes occur only at half-year boundaries. Full monthly extraction stops on missing activity dates; it never silently drops a month or changes provider mid-run.
- HOME uses the selected activity provider, US country partition, the nine California Bay counties, confidence ≥0.60, and the 10th of each month. Optional [10,24] snapshots average sample counts and use the latest eligible HOME per visitor-month. One monthly Bay HOME scan supplies both corridor assignments and Bay sample denominators.
- HOME codes are Census block groups, with fixed TIGER2019 geography based on the pilots. Origin categories: Oakland, San Francisco, Rest East Bay, Rest Bay Area, plus unmatched Bay codes and **No matched Bay Area HOME**. The latter does not imply an outside-Bay residence. Distances use CBG internal points to corridor centroids, not exact homes.
- Raw activity includes all visitors. Adjusted activity includes only matched Bay HOME devices with weights. It is not an estimate of total population visits.
- Sample weight = reference/current HOME sample, within each continuous provider segment. Reference is the first up to 12 months of that segment, with absent CBG samples zero-filled. CBGs with reference or current sample <30 use a Bay-wide ratio from the same segment/provider. Extreme weights remain flagged rather than silently clipped. Short references are explicitly labelled by reference_months.
- **Provider segments are not calibrated to each other.** Do not join adjusted levels into a seamless trend or interpret a provider switch as real activity change. Reference periods can differ in seasonality. Changing HOME assignment rates and selection bias are not solved by sample weights. No extra overlap calibration scans are included.
- Repeats are unweighted, ≥2 dates per corridor/period/half-year. Never deduplicate across six-month resets. Half-year origin uniques may appear in more than one CBG if HOME changes; they are not additive across CBGs.

| Table | Grain / contents |
|---|---|
| corridor_stops | Corridor/month/period; raw stops, visits, unique devices, Bay HOME match and adjusted known-HOME activity |
| corridor_repeat_visitors | Corridor/half-year/period; repeat counts and proportions |
| corridor_homes_monthly | Corridor/month/period/CBG, including unmatched HOME; raw and adjusted counts |
| corridor_homes | Corridor/half-year/period/CBG; half-year uniques and adjusted device-months |
| corridor_origin_categories_monthly | Corridor/month/period/origin category |
| corridor_home_distance_monthly | Corridor/month/period/distance band, including unknown |
| home_sample_index / bay_month_denoms | Monthly CBG weights and Bay HOME sample denominators; not Bay-wide stop totals |
| provider_schedule / provider_coverage_screen | Selected provider/segment/reference and sampled candidate coverage |
| provider_activity_daily / provider_home_sample | Selected-provider daily activity and HOME sample counts |
| home_geography / corridor_geography | Coordinates and geography classifications |
| extraction_coverage / home_month_qa / run_config | Coverage and reproducibility; cover diagnostics exported separately |

All activity tables include provider and segment metadata. Outputs are Snowflake tables plus grouped gzip CSVs without device IDs. Internal checkpoint tables contain device IDs and remain within your working schema.

## Runtime and restart behaviour
No new standalone pilot. Coverage screening is part of this run and cached. Allow roughly **30–120 minutes** for screening, **10–40 hours** for stops (unmeasured monthly/single-provider configuration), **30 minutes–3 hours** for Bay HOME, and **30 minutes–2 hours** for geography, aggregation and exports. A broad first-run budget is **12–48 hours**, not an eight-hour promise. Monthly execution should reduce repeated work, but actual warehouse performance determines the gain. Inspect query_timings.csv after the first production months; those months remain completed results.

Every completed month is checkpointed in Snowflake. Restart and Run All to resume under identical settings. An unfinished month may need to rerun its source query. Do not run two copies simultaneously or change configuration mid-run. Final output tables are saved in a schema suffixed with the run ID; earlier runs remain intact. An error remains an error; it is never converted into a zero result.

## Working in sessions of up to six hours

Long `CREATE TABLE AS SELECT` queries are submitted with Snowflake's asynchronous API and `ABORT_DETACHED_QUERY=FALSE`. Once **Submitted: query-ID → table** appears, that query can complete and save its table without Python staying alive. Closing the computer does **not** start subsequent notebook steps; those resume next session. Warehouse timeouts or administrator cancellation can still stop a query.

The default 5.5-hour session budget stops the notebook before launching more work and leaves any active asynchronous query running. This deliberately raises `SessionPause`; it is a normal stopping point. You can also interrupt Python's polling after the Submitted message. Do not cancel the query in Snowflake. Local geography work and CSV export need Python; rerun them if interrupted.

**Resume:** restart the kernel, keep RUN_LABEL and inputs unchanged, then Run All. Setup reconnects; completed tables are skipped; a submitted query is reattached via the durable `WORK_SCHEMA.EB_ASYNC_JOBS` registry. Completed queries are not resubmitted. Only one notebook instance should orchestrate a run at a time. A rare ambiguous submission stops for query-history inspection instead of risking duplicate work. The registry is authoritative even if local files are lost. If history has expired before a missing query ID can be recovered, manually reconcile the job record and target table before retrying.

**New corridors / fresh run:** change RUN_LABEL (e.g. `expanded_corridors_01`), use a new EXISTING_COVER table name and matching COVER_CSV, and set CORRIDOR_PATH to the matching boundaries. Reusing an existing cover table ignores COVER_CSV, so do not point a new CSV at the old cover-table name. Corridor count is detected dynamically; the coverage threshold remains ≥0.8. The cover content fingerprint and boundary-file hash are part of the run ID. Working and final output schemas are run-specific, so previous results remain intact. No cleanup deletes earlier runs.

**Connection prerequisite:** the notebook looks for a native Snowflake connection exposed by Cuebiq or its SQLAlchemy engine. If unavailable, it stops before long work and asks you to set ASYNC_CONNECTION to the authenticated native connection supplied by your environment. It does not silently fall back to synchronous queries, which may be canceled on disconnect. This wrapper integration cannot be verified outside your Cuebiq environment.

Reference: https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-example#performing-an-asynchronous-query


## 1. Connection and configuration

In [1]:
import time, json, hashlib, re, calendar, struct, zipfile, csv
from pathlib import Path
from datetime import date, timedelta
from urllib.request import urlopen, Request
import pandas as pd
from IPython.display import display

%load_ext cuebiqmagic.magics
%init_cuebiq_data
snow_engine=get_ipython().user_ns['instance']


schema : args.schema='paas_cda_pe_v3' and database: args.database='cuebiq_data' used
The connection to the Snowflake cluster has been established using database 'cuebiq_data' and schema 'paas_cda_pe_v3'.

You can now retrieve the connection using the following statement:
snow_engine = get_ipython().user_ns['instance']

The connection offers two methods


snow_engine.execute_statement("<SQL Statement>") # execute the SQL statement.


snow_engine.read_sql("<SQL Statement>") # execute the SQL statement and return a Pandas DataFrame.


snow_engine.write_dataframe(dataframe:pd.DataFrame, table_name:str,schema:str, index_in=False, if_exists_in="append",method_in="multi")


In [2]:
DATE_START,DATE_END=20190101,20251231
PROVIDERS=['PAPA','LIMA','SIERRA']  # tie priority; all contribute to the pooled panel
COUNTRY='US'
SNAPSHOT_DAYS=[10]                 # [10,24] doubles selected HOME dates
CHUNK_DAYS=31
MAX_DWELL=480
HOME_CONFIDENCE=0.60
MIN_BASELINE_SAMPLE=30
SOURCE_SCHEMA='CUEBIQ_DATA.PAAS_CDA_PE_V3'
STOP_SOURCE=SOURCE_SCHEMA+'.STOP_BY_EVENT_DATE_UPLEVELLED'
HOME_SOURCE=SOURCE_SCHEMA+'.DEVICE_RECURRING_AREA'
WORK_SCHEMA='DEDICATED.EDDIT_EAST_BAY_TMP'
OUT_SCHEMA='DEDICATED.EDDIT_EAST_BAY'
EXISTING_COVER=WORK_SCHEMA+'.TMP_EB_P2_CORRIDOR_GEOHASH9'
COVER_CSV=Path('./corridors/geohash_level9_oak_bids.csv')
CORRIDOR_PATH=Path('./corridors/OAK_BIDs')
CORRIDOR_ID_COLUMN='FID'
CORRIDOR_NAME_COLUMN='BID'
CENSUS_DIR=Path('./geo/census')
EXPORT_ROOT=Path('./activity/east_bay_full')
RUN_DISTANCE=True
EXPORT_CSV=True
CODE_VERSION='east_bay_full_v2_monthly_bay_single'
BAY_COUNTIES=['001','013','041','055','075','081','085','095','097']
EAST_BAY_COUNTIES=['001','013']
# Official state abbreviation/FIPS mapping; preserve leading zeroes in geography codes.
STATE_FIPS=dict(x.split(':') for x in (
 'AL:01 AK:02 AZ:04 AR:05 CA:06 CO:08 CT:09 DE:10 DC:11 FL:12 GA:13 '
 'HI:15 ID:16 IL:17 IN:18 IA:19 KS:20 KY:21 LA:22 ME:23 MD:24 MA:25 MI:26 '
 'MN:27 MS:28 MO:29 MT:30 NE:31 NV:32 NH:33 NJ:34 NM:35 NY:36 NC:37 ND:38 '
 'OH:39 OK:40 OR:41 PA:42 RI:44 SC:45 SD:46 TN:47 TX:48 UT:49 VT:50 VA:51 '
 'WA:53 WV:54 WI:55 WY:56 AS:60 GU:66 MP:69 PR:72 VI:78').split())
assert DATE_START==20190101 and DATE_END==20251231, 'This contract is the complete 2019–2025 run.'
assert SNAPSHOT_DAYS and len(set(SNAPSHOT_DAYS))==len(SNAPSHOT_DAYS) and all(1<=x<=28 for x in SNAPSHOT_DAYS)
assert CHUNK_DAYS>=1
EXPORT_ROOT.mkdir(parents=True,exist_ok=True)
TIMING_FILE=EXPORT_ROOT/'query_timings.csv'

# Optional fixed half-year overrides, e.g. {"2023H1":"LIMA"}; default uses coverage evidence.
PROVIDER_OVERRIDES={}

# Change this label for a completely fresh run; keep it unchanged to resume.
RUN_LABEL='bay_monthly_v3'
SESSION_HOURS=5.5
POLL_SECONDS=15
# Optional: an authenticated snowflake.connector connection, if Cuebiq's wrapper
# does not expose one. Never put passwords in this notebook.
ASYNC_CONNECTION=None
CORRIDOR_ID_PREFIX='Oakland BID:'
SESSION_STARTED=time.monotonic()


## 2. Helpers, cached provider coverage screening, and production checkpoints

The selection scan retains only aggregated counts; it does not pool candidate activity into the production results.

In [3]:
def q(value): return "'"+str(value).replace("'","''")+"'"
def sql_list(values): return '('+','.join(q(v) for v in values)+')'
def read(sql):
    out=snow_engine.read_sql(sql)
    out.columns=[str(c).lower() for c in out.columns]
    return out

class SessionPause(RuntimeError): pass

def session_budget():
    if time.monotonic()-SESSION_STARTED >= SESSION_HOURS*3600:
        raise SessionPause('Session budget reached. An already-submitted async query continues. Restart kernel and Run All next session with unchanged settings.')

# Resolve only known connection/engine attributes; no credential discovery.
def native_connection(wrapper):
    if ASYNC_CONNECTION is not None: return ASYNC_CONNECTION
    candidates=[wrapper]
    for name in ['connection','conn','engine','_connection','_conn','_engine']:
        item=getattr(wrapper,name,None)
        if item is not None: candidates.append(item)
    for obj in candidates:
        if callable(getattr(obj,'get_query_status',None)) and callable(getattr(obj,'cursor',None)):
            return obj
        if callable(getattr(obj,'raw_connection',None)):
            handle=obj.raw_connection()
            raw=getattr(handle,'driver_connection',None)
            if raw is None: raw=getattr(handle,'connection',None)
            if callable(getattr(raw,'get_query_status',None)):
                globals()['_RAW_HANDLE']=handle  # keep SQLAlchemy checkout alive
                return raw
            handle.close()
    raise RuntimeError('Cuebiq wrapper does not expose a native Snowflake connection. Set ASYNC_CONNECTION to your environment’s authenticated snowflake.connector connection, then rerun. No long query was submitted; synchronous execution cannot guarantee survival after disconnect.')

ASYNC_CONN=native_connection(snow_engine)
# All metadata and queries use the same authenticated session/context.
def read(sql):
    cur=ASYNC_CONN.cursor()
    try:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(),columns=[d[0].lower() for d in cur.description])
    finally: cur.close()
read('ALTER SESSION SET ABORT_DETACHED_QUERY=FALSE')
read(f'CREATE SCHEMA IF NOT EXISTS {WORK_SCHEMA}')
JOBS=WORK_SCHEMA+'.EB_ASYNC_JOBS'
read(f"""CREATE TABLE IF NOT EXISTS {JOBS} (
 job_key VARCHAR, query_marker VARCHAR, query_id VARCHAR, target_table VARCHAR,
 submitted_at TIMESTAMP_LTZ, finished_at TIMESTAMP_LTZ, status VARCHAR)""")

def run(label,sql,quiet=False):
    session_budget()
    start=time.monotonic()
    if not quiet: print('Start:',label,flush=True)
    # Long materializations are asynchronous CTAS, including final outputs.
    is_ctas=bool(re.match(r'CREATE(?: OR REPLACE)? TABLE .*? AS ',sql,re.I|re.S))
    if not is_ctas: return read(sql)
    target=re.match(r'CREATE(?: OR REPLACE)? TABLE (\S+)',sql,re.I).group(1)
    key=hashlib.sha256((RUN_LABEL+'|'+sql).encode()).hexdigest()
    jobs=read(f'SELECT * FROM {JOBS} WHERE job_key={q(key)} ORDER BY submitted_at DESC')
    query_id=None; marker=None
    if len(jobs):
        job=jobs.iloc[0]; marker=job.query_marker
        if job.status=='SUCCEEDED' and exists(target): return pd.DataFrame({'status':['REUSED']})
        if job.status in ['SUBMITTED','SUBMITTING']:
            query_id=job.query_id if pd.notna(job.query_id) else None
            if query_id is None:
                # Recover the small crash window between server acceptance and ID recording.
                hist=read(f"""SELECT query_id FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(RESULT_LIMIT=>10000))
                 WHERE STARTSWITH(query_text,{q(marker)}) ORDER BY start_time DESC""")
                if hist.empty:
                    raise RuntimeError(f'Uncertain submission for {target}. Inspect EB_ASYNC_JOBS and Snowflake query history before retrying; no duplicate was submitted. Marker: {marker}')
                query_id=hist.iloc[0].query_id
    if query_id is None:
        import uuid
        marker='/* EB_ASYNC '+uuid.uuid4().hex+' */'
        read(f"INSERT INTO {JOBS}(job_key,query_marker,target_table,submitted_at,status) VALUES ({q(key)},{q(marker)},{q(target)},CURRENT_TIMESTAMP(),'SUBMITTING')")
        cursor=ASYNC_CONN.cursor()
        # If submission raises, leave SUBMITTING for explicit recovery; never blindly retry.
        cursor.execute_async(marker+'\n'+sql)
        query_id=cursor.sfqid
        globals()['_ACTIVE_ASYNC_CURSOR']=cursor
        read(f"UPDATE {JOBS} SET query_id={q(query_id)},status='SUBMITTED' WHERE query_marker={q(marker)}")
        print('Submitted:',query_id,'→',target,flush=True)
    else:
        print('Reconnect to query:',query_id,'→',target,flush=True)
    while True:
        status=ASYNC_CONN.get_query_status(query_id)
        if not ASYNC_CONN.is_still_running(status): break
        session_budget()
        time.sleep(POLL_SECONDS)
    try:
        ASYNC_CONN.get_query_status_throw_if_error(query_id)
        if not exists(target): raise RuntimeError('Query finished without expected table: '+target)
    except Exception:
        read(f"UPDATE {JOBS} SET status='FAILED',finished_at=CURRENT_TIMESTAMP() WHERE query_marker={q(marker)}")
        raise
    read(f"UPDATE {JOBS} SET status='SUCCEEDED',finished_at=CURRENT_TIMESTAMP() WHERE query_marker={q(marker)}")
    seconds=round(time.monotonic()-start,1)
    pd.DataFrame([dict(timestamp=pd.Timestamp.now().isoformat(),run_id=globals().get('RUN_ID',RUN_LABEL),step=label,seconds=seconds,status='OK',query_id=query_id)]).to_csv(TIMING_FILE,mode='a',header=not TIMING_FILE.exists(),index=False)
    if not quiet: print(f'OK: {label} ({seconds:.1f}s observed this session)',flush=True)
    return pd.DataFrame({'status':['OK']})

def exists(table):
    try: read(f'SELECT 1 FROM {table} LIMIT 0'); return True
    except Exception as e:
        if 'does not exist' in str(e).lower() or 'not exist' in str(e).lower(): return False
        raise

def build(table,select_sql,label=None):
    if not exists(table): run(label or table.rsplit('.',1)[-1],f'CREATE TABLE {table} AS '+select_sql)
    return table

def upload_frame(frame,table,types):
    if exists(table): return
    stage=table+'_LOADING'
    run('Initialize lookup '+table.rsplit('.',1)[-1],f'CREATE OR REPLACE TABLE {stage} ('+
        ','.join(k+' '+v for k,v in types.items())+')',quiet=True)
    for start in range(0,len(frame),4000):
        values=[]
        for row in frame.iloc[start:start+4000][list(types)].itertuples(index=False,name=None):
            values.append('('+','.join('NULL' if pd.isna(v) else q(v) for v in row)+')')
        run('Upload lookup batch',f'INSERT INTO {stage} VALUES '+','.join(values),quiet=True)
    actual=int(read(f'SELECT COUNT(*) AS n FROM {stage}').iloc[0,0])
    assert actual==len(frame),(table,actual,len(frame))
    run('Commit lookup '+table.rsplit('.',1)[-1],f'ALTER TABLE {stage} RENAME TO {table}',quiet=True)

run('Ensure working schema',f'CREATE SCHEMA IF NOT EXISTS {WORK_SCHEMA}')
run('Ensure output schema',f'CREATE SCHEMA IF NOT EXISTS {OUT_SCHEMA}')
if not exists(EXISTING_COVER):
    cover=pd.read_csv(COVER_CSV,dtype={'corridor_id':str,'geohash9':str})
    cover['corridor_id']=CORRIDOR_ID_PREFIX+cover.corridor_id
    upload_frame(cover,EXISTING_COVER,{'corridor_id':'VARCHAR','corridor_name':'VARCHAR',
                                    'geohash9':'VARCHAR','coverage_fraction':'FLOAT'})
cover_check=read(f"""SELECT COUNT(*) AS n,COUNT(DISTINCT corridor_id) AS corridors,
 MIN(coverage_fraction) AS min_coverage,
 HASH_AGG(corridor_id,corridor_name,geohash9,coverage_fraction) AS fingerprint
 FROM {EXISTING_COVER}""")
N_CORRIDORS=int(cover_check.iloc[0].corridors)
assert N_CORRIDORS>0 and int(cover_check.iloc[0]['n'])>0
assert float(cover_check.iloc[0].min_coverage)>=0.8
assert int(read(f'SELECT COUNT(*) FROM (SELECT corridor_id,geohash9 FROM {EXISTING_COVER} GROUP BY 1,2 HAVING COUNT(*)>1)').iloc[0,0])==0
# Provider coverage screening is cached independently from production tables.
# One local day/month, all candidate providers in one scan, all corridors together.
probe_config=dict(run_label=RUN_LABEL,version=CODE_VERSION,providers=PROVIDERS,cover=str(cover_check.iloc[0].fingerprint),dwell=MAX_DWELL)
probe_id=hashlib.sha256(json.dumps(probe_config,sort_keys=True).encode()).hexdigest()[:10].upper()
PROBE_PREFIX=WORK_SCHEMA+'.EBSEL_'+probe_id+'_'
months=list(pd.period_range('2019-01','2025-12',freq='M'))
month_ids=[int(str(m).replace('-','')) for m in months]
probe_tables=[]
for m in months:
    ym=int(str(m).replace('-','')); date=m.start_time+pd.Timedelta(days=9)
    target=int(date.strftime('%Y%m%d')); probe=PROBE_PREFIX+str(ym);probe_tables.append(probe)
    build(probe,f"""SELECT {ym} AS year_month,s.provider_id,
      COUNT(DISTINCT s.cuebiq_id) AS devices,COUNT(DISTINCT c.corridor_id) AS corridors
      FROM {STOP_SOURCE} s JOIN {EXISTING_COVER} c ON s.geohash_id=c.geohash9
      WHERE s.provider_id IN {sql_list(PROVIDERS)} AND s.country_code='{COUNTRY}'
       AND s.event_date BETWEEN {int((date-pd.Timedelta(days=1)).strftime('%Y%m%d'))}
                            AND {int((date+pd.Timedelta(days=1)).strftime('%Y%m%d'))}
       AND s.dwell_time_minutes>0 AND s.dwell_time_minutes<={MAX_DWELL}
       AND TO_NUMBER(TO_CHAR(TO_TIMESTAMP_TZ(s.stop_zoned_datetime),'YYYYMMDD'))={target}
      GROUP BY s.provider_id""",f'Provider coverage {ym}')
provider_checks=read(' UNION ALL '.join('SELECT * FROM '+t for t in probe_tables))
provider_checks['half_year']=provider_checks.year_month.map(lambda m:f'{int(m)//100}H{1 if int(m)%100<=6 else 2}')
schedule=[]
for year in range(2019,2026):
    for half in [1,2]:
        label=f'{year}H{half}'
        mids=[m for m in month_ids if m//100==year and (1 if m%100<=6 else 2)==half]
        sub=provider_checks.loc[provider_checks.half_year.eq(label)]
        scores=sub.groupby('provider_id').agg(months_observed=('year_month','nunique'),
                   sampled_devices=('devices','sum'),min_corridors=('corridors','min'))
        eligible=scores.loc[scores.months_observed.eq(6)].sort_values('sampled_devices',ascending=False)
        if eligible.empty: raise ValueError(f'{label}: no provider has activity on all six sampled dates. Review provider_checks; do not silently discard months.')
        provider=PROVIDER_OVERRIDES.get(label,eligible.index[0])
        if provider not in eligible.index: raise ValueError(f'{label}: override lacks six-month sampled coverage.')
        for ym in mids: schedule.append(dict(year_month=ym,half_year=label,provider_id=provider))
schedule=pd.DataFrame(schedule)
schedule['segment_number']=schedule.provider_id.ne(schedule.provider_id.shift()).cumsum()
schedule['provider_segment']=schedule.apply(lambda r:f"S{r.segment_number}_{r.provider_id}",axis=1)
schedule['reference_start_month']=schedule.groupby('provider_segment').year_month.transform('min')
# First up to 12 months within each continuous provider segment; never bridge providers.
refs={seg:list(g.year_month.iloc[:12]) for seg,g in schedule.groupby('provider_segment',sort=False)}
schedule['is_reference_month']=schedule.apply(lambda r:int(r.year_month in refs[r.provider_segment]),axis=1)
schedule['reference_months']=schedule.provider_segment.map(lambda seg:len(refs[seg]))
schedule['selection_basis']='largest_sum_of_monthly_sample_day_devices_among_full_half_year_candidates'
provider_for_month=dict(zip(schedule.year_month,schedule.provider_id))
display(schedule[['half_year','provider_id','provider_segment','reference_start_month','reference_months']].drop_duplicates())
boundary_files=sorted(CORRIDOR_PATH.glob('*')) if CORRIDOR_PATH.is_dir() else sorted(CORRIDOR_PATH.parent.glob(CORRIDOR_PATH.stem+'.*'))
boundary_hash=hashlib.sha256(b''.join(f.name.encode()+f.read_bytes() for f in boundary_files if f.is_file())).hexdigest()
config=dict(boundary_hash=boundary_hash,corridor_prefix=CORRIDOR_ID_PREFIX,run_label=RUN_LABEL,version=CODE_VERSION,start=DATE_START,end=DATE_END,providers=PROVIDERS,country=COUNTRY,
 snapshot_days=SNAPSHOT_DAYS,chunk_days=CHUNK_DAYS,dwell=MAX_DWELL,confidence=HOME_CONFIDENCE,
 baseline_min=MIN_BASELINE_SAMPLE,cover_hash=str(cover_check.iloc[0].fingerprint),geography='TIGER2019',
 distance=RUN_DISTANCE,provider_identity='single_per_half_year',baseline='segment_first_up_to_12_months',
 provider_schedule=schedule.to_dict('records'),home_scope='nine_county_bay')
RUN_ID=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:10].upper()
P=WORK_SCHEMA+'.EBF_'+RUN_ID+'_'
BASE_OUT_SCHEMA=OUT_SCHEMA
OUT_SCHEMA=BASE_OUT_SCHEMA+'_'+RUN_ID
run('Ensure run output schema',f'CREATE SCHEMA IF NOT EXISTS {OUT_SCHEMA}')
EXPORT_DIR=EXPORT_ROOT/RUN_ID
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
(EXPORT_DIR/'run_config.json').write_text(json.dumps(config,indent=2))
months=list(pd.period_range('2019-01','2025-12',freq='M'))
month_ids=[int(str(m).replace('-','')) for m in months]
def tab(kind,month): return P+kind+'_'+str(month)
def union_tables(tables): return ' UNION ALL '.join('SELECT * FROM '+t for t in tables)
def month_bounds(m): return int(m.start_time.strftime('%Y%m%d')),int(m.end_time.strftime('%Y%m%d'))
def dt(v): return pd.Timestamp(str(v)).date()
def num(d): return int(d.strftime('%Y%m%d'))
provider_sql=sql_list(PROVIDERS)
print('Run ID:',RUN_ID,'Months:',len(months),'HOME dates:',len(months)*len(SNAPSHOT_DAYS))
print('Completed configuration-matching tables will be reused.')

SCHEDULE=P+'PROVIDER_SCHEDULE'
upload_frame(schedule,SCHEDULE,{'year_month':'NUMBER','half_year':'VARCHAR','provider_id':'VARCHAR','provider_segment':'VARCHAR','reference_start_month':'NUMBER','is_reference_month':'NUMBER','reference_months':'NUMBER','selection_basis':'VARCHAR'})

print('Output schema for this run:',OUT_SCHEMA)
display(read(f"SELECT target_table,query_id,status,submitted_at,finished_at FROM {JOBS} WHERE target_table LIKE {q(P+'%')} ORDER BY submitted_at DESC LIMIT 20"))


Start: Ensure working schema
Start: Ensure output schema


,half_year,provider_id,provider_segment,reference_start_month,reference_months
0,2019H1,LIMA,S1_LIMA,201901,12
6,2019H2,LIMA,S1_LIMA,201901,12
12,2020H1,LIMA,S1_LIMA,201901,12
18,2020H2,LIMA,S1_LIMA,201901,12
24,2021H1,LIMA,S1_LIMA,201901,12
30,2021H2,LIMA,S1_LIMA,201901,12
36,2022H1,LIMA,S1_LIMA,201901,12
42,2022H2,LIMA,S1_LIMA,201901,12
48,2023H1,SIERRA,S2_SIERRA,202301,6
54,2023H2,PAPA,S3_PAPA,202307,12


Start: Ensure run output schema
Run ID: 70E3F58D7C Months: 84 HOME dates: 84
Completed configuration-matching tables will be reused.
Output schema for this run: DEDICATED.EDDIT_EAST_BAY_70E3F58D7C


,target_table,query_id,status,submitted_at,finished_at
0,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_R_...,01c7785f-010c-7a8e-0002-4a7603f3938a,SUBMITTED,2026-10-02 15:55:17.719000-07:00,NaT
1,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_DM...,01c7785f-010c-81f3-0002-4a7603f34932,SUCCEEDED,2026-10-02 15:54:59.967000-07:00,2026-10-02 15:55:16.312000-07:00
2,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_CO...,01c7785e-010c-7a8e-0002-4a7603f3935e,SUCCEEDED,2026-10-02 15:54:43.551000-07:00,2026-10-02 15:54:59.400000-07:00
3,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_PR...,01c7785e-010c-7a8e-0002-4a7603f39352,SUCCEEDED,2026-10-02 15:54:26.755000-07:00,2026-10-02 15:54:42.938000-07:00
4,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_CQ...,01c7785e-010c-7a8e-0002-4a7603f39342,SUCCEEDED,2026-10-02 15:54:09.904000-07:00,2026-10-02 15:54:26.025000-07:00
5,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_D_...,01c7785d-010c-7a8e-0002-4a7603f39332,SUCCEEDED,2026-10-02 15:53:52.738000-07:00,2026-10-02 15:54:08.730000-07:00
6,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_PQ...,01c7785d-010c-7a8e-0002-4a7603f39326,SUCCEEDED,2026-10-02 15:53:35.780000-07:00,2026-10-02 15:53:52.056000-07:00
7,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_R_...,01c77846-010c-81f3-0002-4a7603f348d2,SUCCEEDED,2026-10-02 15:30:25.913000-07:00,2026-10-02 15:53:35.120000-07:00
8,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_DM...,01c77846-010c-81f3-0002-4a7603f348aa,SUCCEEDED,2026-10-02 15:30:08.420000-07:00,2026-10-02 15:30:24.304000-07:00
9,DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_CO...,01c77845-010c-7a8e-0002-4a7603f392fa,SUCCEEDED,2026-10-02 15:29:51.982000-07:00,2026-10-02 15:30:07.820000-07:00


## 3. Fixed 2019 geography and corridor centroids (cached local work)

In [4]:
def census_zip(state_fips,layer='bg'):
    path=CENSUS_DIR/f'tl_2019_{state_fips}_{layer}.zip'
    CENSUS_DIR.mkdir(parents=True,exist_ok=True)
    if path.exists() and zipfile.is_zipfile(path): return path
    url=f'https://www2.census.gov/geo/tiger/TIGER2019/{layer.upper()}/'+path.name
    print('Download once:',path.name,flush=True)
    partial=path.with_suffix('.part')
    with urlopen(Request(url,headers={'User-Agent':'EastBayResearch/1.0'}),timeout=90) as src, open(partial,'wb') as dst:
        while True:
            chunk=src.read(1024*1024)
            if not chunk: break
            dst.write(chunk)
    if not zipfile.is_zipfile(partial): raise ValueError('Census response was not a ZIP: '+url)
    partial.replace(path)
    return path

# DBF attributes only: no national polygon loading and no extra DBF dependency.
def dbf_attributes(zip_path,wanted):
    with zipfile.ZipFile(zip_path) as z:
        name=next(n for n in z.namelist() if n.lower().endswith('.dbf'))
        with z.open(name) as f:
            h=f.read(32)
            nrec=struct.unpack('<I',h[4:8])[0]; hlen,rlen=struct.unpack('<HH',h[8:12])
            descriptors=f.read(hlen-32)
            fields=[]; offset=1
            for pos in range(0,len(descriptors)-1,32):
                d=descriptors[pos:pos+32]
                if len(d)<32 or d[0]==13: break
                name=d[:11].split(b'\0',1)[0].decode('ascii'); length=d[16]
                if name in wanted: fields.append((name,offset,length))
                offset+=length
            rows=[]
            for _ in range(nrec):
                record=f.read(rlen)
                if not record: break
                if record[:1]==b'*': continue
                rows.append({name:record[o:o+l].decode('latin1').strip() for name,o,l in fields})
    return pd.DataFrame(rows)

BAY_LOOKUP=P+'BAY_CBG'
CORRIDOR_GEO=P+'CORRIDOR_GEO'
if not exists(BAY_LOOKUP) or not exists(CORRIDOR_GEO):
    import geopandas as gpd
if not exists(BAY_LOOKUP):
    bg=gpd.read_file(census_zip('06'),bbox=(-124.0,36.5,-121.0,39.5))
    bg['geoid']=bg.GEOID.astype(str); bg['county']=bg.geoid.str[2:5]
    bg=bg.loc[bg.county.isin(BAY_COUNTIES)].copy()
    places=gpd.read_file(census_zip('06','place'),bbox=(-124.0,36.5,-121.0,39.5))
    oak=places.loc[places.NAME.eq('Oakland')].to_crs('EPSG:3310')
    assert len(oak)==1
    inside=bg.to_crs('EPSG:3310').geometry.representative_point().within(oak.geometry.iloc[0])
    bg['home_geography']='Rest Bay Area'
    bg.loc[bg.county.isin(EAST_BAY_COUNTIES),'home_geography']='Rest East Bay'
    bg.loc[bg.county.eq('075'),'home_geography']='San Francisco'
    bg.loc[inside,'home_geography']='Oakland'
    bg['block_group_id']='US.CA.'+bg.geoid.str[2:5]+'.'+bg.geoid.str[5:11]+'.'+bg.geoid.str[11:12]
    upload_frame(bg,BAY_LOOKUP,{'block_group_id':'VARCHAR','home_geography':'VARCHAR'})
if not exists(CORRIDOR_GEO):
    boundary=CORRIDOR_PATH/(CORRIDOR_PATH.name+'.shp') if CORRIDOR_PATH.is_dir() else CORRIDOR_PATH
    if RUN_DISTANCE:
        g=gpd.read_file(boundary)
        assert g.crs is not None and not g.geometry.isna().any() and g.geometry.is_valid.all()
        centers=g.to_crs('EPSG:3310').geometry.centroid.to_crs('EPSG:4326')
        cg=pd.DataFrame({'corridor_id':CORRIDOR_ID_PREFIX+g[CORRIDOR_ID_COLUMN].astype(str),
          'corridor_name':g[CORRIDOR_NAME_COLUMN].astype(str),'corridor_lat':centers.y,'corridor_lon':centers.x})
        cover_names=read(f'SELECT DISTINCT corridor_id,corridor_name FROM {EXISTING_COVER}')
        assert set(map(tuple,cg[['corridor_id','corridor_name']].to_numpy()))==set(map(tuple,cover_names.to_numpy()))
    else:
        cg=read(f'SELECT DISTINCT corridor_id,corridor_name FROM {EXISTING_COVER}')
        cg['corridor_lat']=None;cg['corridor_lon']=None
    upload_frame(cg,CORRIDOR_GEO,{'corridor_id':'VARCHAR','corridor_name':'VARCHAR','corridor_lat':'FLOAT','corridor_lon':'FLOAT'})
print('Geography ready. All years use TIGER2019.')


Geography ready. All years use TIGER2019.


## 4. Monthly stop extraction and device-month aggregation

One selected-provider scan per calendar month, all corridors together. Event dates are padded by one day for local-time boundaries, then filtered to the exact local month. The padding is deduplicated. Completed monthly tables are reused; intermediates are dropped after commit.

In [5]:
all_provider_tables=[];all_coverage_tables=[]
for month in months:
    ym=int(str(month).replace('-','')); first,last=month_bounds(month)
    provider_sql=sql_list([provider_for_month[ym]])
    dm=tab('DM',ym); pm=tab('PROVIDER_DAY',ym); cm=tab('COVERAGE',ym)
    if exists(dm) and exists(pm) and exists(cm):
        if int(read(f'SELECT COUNT(*) FROM {dm}').iloc[0,0])==0:
            raise ValueError(f'Empty completed month {ym}; inspect coverage before continuing.')
        check=read(f'SELECT SUM(requested_days),SUM(days_with_activity) FROM {cm}').iloc[0]
        if int(check.iloc[0])!=int(check.iloc[1]): raise ValueError(f'Missing activity dates in {ym}; inspect coverage.')
        print('Reuse month',ym);continue
    daily=[];pqa=[];cqa=[];raw_tables=[]
    start=dt(first)
    while start<=dt(last):
        end=min(start+timedelta(days=CHUNK_DAYS-1),dt(last)); a,b=num(start),num(end)
        suffix=str(a)+'_'+str(b)
        raw=P+'R_'+suffix; day=P+'D_'+suffix; pq=P+'PQ_'+suffix; cq=P+'CQ_'+suffix
        daily.append(day);pqa.append(pq);cqa.append(cq);raw_tables.append(raw)
        if not (exists(day) and exists(pq) and exists(cq)):
            build(raw,f"""
            SELECT c.corridor_id,c.corridor_name,s.provider_id,s.cuebiq_id,s.geohash_id,
              TO_TIMESTAMP_TZ(s.stop_zoned_datetime) AS stop_start_ts,s.dwell_time_minutes,
              TO_NUMBER(TO_CHAR(TO_TIMESTAMP_TZ(s.stop_zoned_datetime),'YYYYMMDD')) AS visit_date,
              DAYOFWEEKISO(TO_TIMESTAMP_TZ(s.stop_zoned_datetime)) AS dow_iso,
              HOUR(TO_TIMESTAMP_TZ(s.stop_zoned_datetime)) AS hour_of_day
            FROM {STOP_SOURCE} s JOIN {EXISTING_COVER} c ON s.geohash_id=c.geohash9
            WHERE s.provider_id IN {provider_sql} AND s.country_code='{COUNTRY}'
              AND s.event_date BETWEEN {num(start-timedelta(days=1))} AND {num(end+timedelta(days=1))}
              AND s.stop_zoned_datetime IS NOT NULL AND s.dwell_time_minutes>0
              AND s.dwell_time_minutes<={MAX_DWELL}
              AND TO_NUMBER(TO_CHAR(TO_TIMESTAMP_TZ(s.stop_zoned_datetime),'YYYYMMDD')) BETWEEN {a} AND {b}
            QUALIFY ROW_NUMBER() OVER (PARTITION BY s.provider_id,c.corridor_id,s.cuebiq_id,
              s.geohash_id,s.stop_zoned_datetime,s.dwell_time_minutes
              ORDER BY s.processing_date DESC,s.event_date ASC)=1
            """,f'Source stops {a}–{b}')
            build(pq,f"""SELECT {ym} AS year_month,visit_date,provider_id,
              COUNT(*) AS corridor_stop_assignments,COUNT(DISTINCT cuebiq_id) AS distinct_ids_all_corridors
              FROM {raw} GROUP BY 2,3""",f'Provider QA {a}')
            build(day,f"""
            WITH pooled AS (SELECT * FROM {raw}), periods AS (
              SELECT *, 'all' AS time_period FROM pooled
              UNION ALL SELECT *, 'weekdays' FROM pooled WHERE dow_iso BETWEEN 1 AND 5
              UNION ALL SELECT *, 'weekends' FROM pooled WHERE dow_iso IN (6,7)
              UNION ALL SELECT *, 'nine-five' FROM pooled WHERE dow_iso BETWEEN 1 AND 5 AND hour_of_day BETWEEN 9 AND 16
              UNION ALL SELECT *, 'evening' FROM pooled WHERE hour_of_day BETWEEN 17 AND 22
            )
            SELECT corridor_id,corridor_name,cuebiq_id,visit_date,time_period,
                   COUNT(*) AS raw_stop_count,1 AS visit_count
            FROM periods GROUP BY 1,2,3,4,5
            """,f'Device-days {a}–{b}')
            build(cq,f"""SELECT {ym} AS year_month,{a} AS chunk_start,{b} AS chunk_end,
              {int((end-start).days+1)} AS requested_days,
              COUNT(DISTINCT visit_date) AS days_with_activity,COUNT(DISTINCT corridor_id) AS corridors,
              COALESCE(SUM(raw_stop_count),0) AS raw_stops,COUNT(*) AS corridor_device_days
              FROM {day} WHERE time_period='all'""",f'Chunk QA {a}')
        start=end+timedelta(days=1)
    build(pm,union_tables(pqa),f'Commit provider month {ym}')
    build(cm,union_tables(cqa),f'Commit coverage {ym}')
    build(dm,f"""SELECT corridor_id,corridor_name,cuebiq_id,{ym} AS year_month,time_period,
       SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count
       FROM ({union_tables(daily)}) GROUP BY 1,2,3,5""",f'Commit device-month {ym}')
    if int(read(f'SELECT COUNT(*) FROM {dm}').iloc[0,0])==0:
        raise ValueError(f'No corridor activity in {ym}; inspect provider/date coverage before continuing.')
    check=read(f'SELECT SUM(requested_days),SUM(days_with_activity) FROM {cm}').iloc[0]
    if int(check.iloc[0])!=int(check.iloc[1]): raise ValueError(f'Missing activity dates in {ym}; inspect coverage before continuing.')
    for table in raw_tables+daily+pqa+cqa:
        run('Release completed chunk',f'DROP TABLE IF EXISTS {table}',quiet=True)
print('All 84 monthly device tables completed.')


Reuse month 201901
Reuse month 201902
Reuse month 201903
Reuse month 201904
Reuse month 201905
Reuse month 201906
Reuse month 201907
Reuse month 201908
Reuse month 201909
Reuse month 201910
Reuse month 201911
Reuse month 201912
Reuse month 202001
Reuse month 202002
Reuse month 202003
Reuse month 202004
Reuse month 202005
Reuse month 202006
Reuse month 202007
Reuse month 202008
Reuse month 202009
Reuse month 202010
Reuse month 202011
Reuse month 202012
Reuse month 202101
Reuse month 202102
Reuse month 202103
Reuse month 202104
Reuse month 202105
Reuse month 202106
Reuse month 202107
Reuse month 202108
Reuse month 202109
Reuse month 202110
Reuse month 202111
Reuse month 202112
Reuse month 202201
Reuse month 202202
Reuse month 202203
Reuse month 202204
Reuse month 202205
Reuse month 202206
Reuse month 202207
Reuse month 202208
Reuse month 202209
Reuse month 202210
Reuse month 202211
Reuse month 202212
Reuse month 202301
Reuse month 202302
Reuse month 202303
Reuse month 202304
Reuse month 

## 5. Monthly Bay Area HOME and sample denominators

Direct provider/country/snapshot partition predicates plus Bay county filters. Geography filters reduce downstream data but may not reduce source bytes as much as partition predicates. The bounded monthly snapshot is reused for all HOME products and then dropped.

In [6]:
priority='CASE h.provider_id '+' '.join(f'WHEN {q(p)} THEN {i}' for i,p in enumerate(PROVIDERS))+' ELSE 99 END'
county_sql=sql_list(BAY_COUNTIES)
for month in months:
    ym=int(str(month).replace('-','')); provider_sql=sql_list([provider_for_month[ym]]); snapshots=[ym*100+d for d in sorted(SNAPSHOT_DAYS)]
    dates_sql='('+','.join(map(str,snapshots))+')'
    snap=tab('SNAP',ym); sm=tab('SAMPLE',ym); den=tab('DEN',ym); hm=tab('HOME',ym); hp=tab('HOME_PROVIDER',ym)
    if all(exists(t) for t in [sm,den,hm,hp]):
        print('Reuse HOME month',ym);continue
    build(snap,f"""
    WITH chosen AS (
      SELECT h.provider_id,h.cuebiq_id,h.snapshot_event_date,h.block_group_id,h.confidence_level
      FROM {HOME_SOURCE} h
      WHERE h.provider_id IN {provider_sql} AND h.country_code='{COUNTRY}'
        AND h.snapshot_event_date IN {dates_sql} AND h.tag_type_code='HOME'
        AND SPLIT_PART(h.block_group_id,'.',1)='US'
        AND SPLIT_PART(h.block_group_id,'.',2)='CA'
        AND SPLIT_PART(h.block_group_id,'.',3) IN {county_sql}
        AND h.confidence_level>={HOME_CONFIDENCE}
        AND h.block_group_id IS NOT NULL AND TRIM(h.block_group_id)<>''
      QUALIFY ROW_NUMBER() OVER(PARTITION BY h.snapshot_event_date,h.cuebiq_id
        ORDER BY h.confidence_level DESC,{priority},h.block_group_id)=1
    ), located AS (
      SELECT h.*,IFF(SPLIT_PART(h.block_group_id,'.',2)='CA' AND
        SPLIT_PART(h.block_group_id,'.',3) IN {county_sql},1,0) AS is_bay,
        CASE
          WHEN NOT REGEXP_LIKE(h.block_group_id,'US[.][A-Z]{{2}}[.][0-9]{{3}}[.][0-9]{{6}}[.][0-9]') THEN 'Unclassified HOME code'
          WHEN SPLIT_PART(h.block_group_id,'.',2)='CA' AND SPLIT_PART(h.block_group_id,'.',3) IN {county_sql}
            THEN COALESCE(b.home_geography,'Unmapped Bay Area')
          WHEN SPLIT_PART(h.block_group_id,'.',2)='CA' THEN 'Rest California'
          ELSE 'Rest US' END AS home_geography
      FROM chosen h LEFT JOIN {BAY_LOOKUP} b ON h.block_group_id=b.block_group_id
    )
    SELECT *,CASE WHEN home_geography='Unclassified HOME code' THEN NULL
      WHEN is_bay=1 THEN 'CBG:'||block_group_id ELSE 'REGION:'||home_geography END AS norm_key
    FROM located
    """,f'Bay HOME snapshots {ym}')
    available=set(read(f'SELECT DISTINCT snapshot_event_date FROM {snap}').snapshot_event_date.astype(int))
    if available!=set(snapshots): raise ValueError(f'Missing confident Bay HOME snapshot(s): {set(snapshots)-available}. Source completeness needs review.')
    ns=len(snapshots)
    build(sm,f"""SELECT {ym} AS year_month,norm_key,MAX(home_geography) AS home_geography,
      MAX(is_bay) AS is_bay,COUNT(*)/{ns}.0 AS current_sample
      FROM {snap} WHERE norm_key IS NOT NULL GROUP BY norm_key""",f'HOME sample counts {ym}')
    build(den,f"""SELECT {ym} AS year_month,{ns} AS snapshots,
      COUNT(*)/{ns}.0 AS bay_confident_home_sample,
      COALESCE(COUNT_IF(is_bay=1),0)/{ns}.0 AS bay_home_sample,
      COALESCE(COUNT_IF(norm_key IS NULL),0)/{ns}.0 AS unclassified_home_sample,
      COUNT(DISTINCT provider_id) AS contributing_providers
      FROM {snap}""",f'HOME denominators {ym}')
    build(hp,f"""SELECT {ym} AS year_month,provider_id,COUNT(*)/{ns}.0 AS selected_home_sample,
      COALESCE(COUNT_IF(is_bay=1),0)/{ns}.0 AS selected_bay_home_sample
      FROM {snap} GROUP BY provider_id""",f'HOME provider mix {ym}')
    build(hm,f"""SELECT {ym} AS year_month,h.cuebiq_id,h.provider_id AS home_provider_id,
       h.block_group_id,h.home_geography,h.norm_key,h.confidence_level,h.snapshot_event_date
       FROM {snap} h JOIN (SELECT DISTINCT cuebiq_id FROM {tab('DM',ym)}) v ON h.cuebiq_id=v.cuebiq_id
       QUALIFY ROW_NUMBER() OVER(PARTITION BY h.cuebiq_id
         ORDER BY h.snapshot_event_date DESC,h.confidence_level DESC,h.block_group_id)=1""",f'Corridor HOME assignments {ym}')
    run('Release monthly Bay snapshot',f'DROP TABLE IF EXISTS {snap}',quiet=True)
print('Monthly HOME and sample counts complete.')


Start: Bay HOME snapshots 201901
Submitted: 01c78818-010c-9cea-0002-4a7603f4cdba → DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_SNAP_201901
OK: Bay HOME snapshots 201901 (17.5s observed this session)
Start: HOME sample counts 201901
Submitted: 01c78818-010c-9cee-0002-4a7603f4db8a → DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_SAMPLE_201901
OK: HOME sample counts 201901 (18.9s observed this session)
Start: HOME denominators 201901
Submitted: 01c78818-010c-9cea-0002-4a7603f4ce56 → DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_DEN_201901
OK: HOME denominators 201901 (19.4s observed this session)
Start: HOME provider mix 201901
Submitted: 01c78819-010c-9cea-0002-4a7603f4cf36 → DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_HOME_PROVIDER_201901
OK: HOME provider mix 201901 (2.2s observed this session)
Start: Corridor HOME assignments 201901
Submitted: 01c78819-010c-9cee-0002-4a7603f4dd0a → DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_HOME_201901
OK: Corridor HOME assignments 201901 (17.5s observed 

## 6. Bay Area CBG internal-point coordinates

Only California attributes are needed; the cached 2019 Census ZIP is reused.

In [7]:
HOME_GEO=P+'HOME_GEOGRAPHY'
if not exists(HOME_GEO):
    observed=read('SELECT DISTINCT block_group_id,home_geography FROM ('+
                  union_tables([tab('HOME',m) for m in month_ids])+')')
    pattern=r'^US\.([A-Z]{2})\.(\d{3})\.(\d{6})\.(\d)$'
    parsed=observed.block_group_id.str.extract(pattern)
    observed['state']=parsed[0]
    observed['geoid']=parsed[0].map(STATE_FIPS)+parsed[1]+parsed[2]+parsed[3]
    observed['home_lat']=float('nan');observed['home_lon']=float('nan')
    if RUN_DISTANCE:
        for state in sorted(observed.state.dropna().unique()):
            if state not in STATE_FIPS:
                print('Unknown state code; retained without coordinates:',state);continue
            attributes=dbf_attributes(census_zip(STATE_FIPS[state]),{'GEOID','INTPTLAT','INTPTLON'})
            if not {'GEOID','INTPTLAT','INTPTLON'}.issubset(attributes):
                raise ValueError('Missing Census internal-point fields for '+state)
            attributes=attributes.drop_duplicates('GEOID').set_index('GEOID')
            mask=observed.state.eq(state)
            observed.loc[mask,'home_lat']=pd.to_numeric(observed.loc[mask,'geoid'].map(attributes.INTPTLAT),errors='coerce')
            observed.loc[mask,'home_lon']=pd.to_numeric(observed.loc[mask,'geoid'].map(attributes.INTPTLON),errors='coerce')
    observed['census_vintage']='TIGER2019'
    observed['coordinate_status']=observed.home_lat.notna().map({True:'matched',False:'unmatched_or_disabled'})
    upload_frame(observed,HOME_GEO,{'block_group_id':'VARCHAR','home_geography':'VARCHAR','geoid':'VARCHAR',
      'home_lat':'FLOAT','home_lon':'FLOAT','census_vintage':'VARCHAR','coordinate_status':'VARCHAR'})
    print('Observed CBG coordinate coverage:',observed.home_lat.notna().mean())


Observed CBG coordinate coverage: 1.0


## 7. Within-provider-segment weights and final outputs

Each continuous provider segment has its own reference. All output activity tables carry the provider, segment, reference start and number of reference months. No cross-provider calibration is implied.

In [8]:
def publish(name,select_sql):
    if name.startswith('corridor_') and name not in ['corridor_geography']:
        key='half_year' if name in ['corridor_homes','corridor_repeat_visitors'] else 'year_month'
        select_sql=f"SELECT a.*,s.provider_id,s.provider_segment,s.reference_start_month,s.reference_months FROM ({select_sql}) a JOIN (SELECT DISTINCT {key},provider_id,provider_segment,reference_start_month,reference_months FROM {SCHEDULE}) s USING({key})"
    run('Publish '+name,f'CREATE OR REPLACE TABLE {OUT_SCHEMA}.{name} AS '+select_sql)
def final(name): return OUT_SCHEMA+'.'+name
sample_all='('+union_tables([tab('SAMPLE',m) for m in month_ids])+')'
den_all='('+union_tables([tab('DEN',m) for m in month_ids])+')'
publish('provider_schedule',f'SELECT * FROM {SCHEDULE}')
publish('provider_coverage_screen', ' UNION ALL '.join('SELECT * FROM '+t for t in probe_tables))
publish('bay_month_denoms',f"""
WITH d AS (SELECT a.*,s.provider_id,s.provider_segment,s.reference_start_month,
 s.reference_months,s.is_reference_month FROM {den_all} a JOIN {SCHEDULE} s USING(year_month)),
b AS (SELECT provider_segment,AVG(bay_home_sample) AS baseline_bay_home_sample
 FROM d WHERE is_reference_month=1 GROUP BY provider_segment)
SELECT d.*,b.baseline_bay_home_sample,baseline_bay_home_sample/NULLIF(bay_home_sample,0) AS bay_sample_ratio,
 'within_segment_only_not_cross_provider_calibrated' AS panel_comparability
 FROM d JOIN b USING(provider_segment)""")
publish('home_sample_index',f"""
WITH samples AS (SELECT * FROM {sample_all}),
 cells AS (SELECT DISTINCT norm_key,home_geography,is_bay FROM samples),
 monthly_grid AS (
 SELECT d.year_month,d.provider_segment,d.is_reference_month,c.norm_key,c.home_geography,c.is_bay,
 COALESCE(s.current_sample,0) AS current_sample
 FROM {final('bay_month_denoms')} d CROSS JOIN cells c
 LEFT JOIN samples s ON s.year_month=d.year_month AND s.norm_key=c.norm_key),
 baseline AS (SELECT provider_segment,norm_key,AVG(current_sample) AS baseline_sample,
 STDDEV_SAMP(current_sample) AS baseline_sd FROM monthly_grid WHERE is_reference_month=1 GROUP BY 1,2),
 weighted AS (
 SELECT m.*,b.baseline_sample,b.baseline_sd/NULLIF(b.baseline_sample,0) AS baseline_cv,
 m.current_sample/NULLIF(b.baseline_sample,0) AS sample_index,
 b.baseline_sample/NULLIF(m.current_sample,0) AS direct_sample_ratio,
 CASE WHEN b.baseline_sample<{MIN_BASELINE_SAMPLE} OR m.current_sample<{MIN_BASELINE_SAMPLE}
 THEN d.bay_sample_ratio ELSE b.baseline_sample/NULLIF(m.current_sample,0) END AS applied_sample_ratio,
 CASE WHEN b.baseline_sample<{MIN_BASELINE_SAMPLE} OR m.current_sample<{MIN_BASELINE_SAMPLE}
 THEN 'bay_fallback' ELSE 'cbg' END AS normalization_method,
 'within_segment_only_not_cross_provider_calibrated' AS panel_comparability
 FROM monthly_grid m JOIN baseline b ON m.provider_segment=b.provider_segment AND m.norm_key=b.norm_key
 JOIN {final('bay_month_denoms')} d ON d.year_month=m.year_month)
 SELECT *,IFF(current_sample<{MIN_BASELINE_SAMPLE},1,0) AS low_current_sample,
 IFF(applied_sample_ratio>5 OR applied_sample_ratio<0.2,1,0) AS extreme_weight FROM weighted""")
# Build one annotated device-month table at a time; no further source scans.
for ym in month_ids:
    dest=tab('ENRICHED',ym)
    build(dest,f"""
    SELECT v.*,COALESCE(h.block_group_id,'UNKNOWN') AS block_group_id,
      COALESCE(h.home_geography,'No matched Bay Area HOME') AS home_geography,h.norm_key,h.home_provider_id,
      h.confidence_level,h.snapshot_event_date,
      s.applied_sample_ratio,COALESCE(s.normalization_method,'no_eligible_home_or_reference') AS normalization_method,
      IFF(h.cuebiq_id IS NOT NULL,1,0) AS has_confident_home,
      CASE WHEN g.home_lat IS NOT NULL AND c.corridor_lat IS NOT NULL
        THEN ST_DISTANCE(ST_MAKEPOINT(g.home_lon,g.home_lat),ST_MAKEPOINT(c.corridor_lon,c.corridor_lat))/1000.0
        ELSE NULL END AS distance_km
    FROM {tab('DM',ym)} v LEFT JOIN {tab('HOME',ym)} h ON v.cuebiq_id=h.cuebiq_id
    LEFT JOIN {final('home_sample_index')} s ON s.year_month=v.year_month AND s.norm_key=h.norm_key
    LEFT JOIN {HOME_GEO} g ON g.block_group_id=h.block_group_id
    JOIN {CORRIDOR_GEO} c ON c.corridor_id=v.corridor_id
    """,f'Enrich month {ym}')
enriched='('+union_tables([tab('ENRICHED',m) for m in month_ids])+')'
# Monthly totals keep raw totals and adjusted eligible-HOME totals visibly separate.
publish('corridor_stops',f"""
WITH a AS (
SELECT corridor_id,corridor_name,year_month,time_period,
 SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count,COUNT(*) AS unique_devices,
 SUM(has_confident_home) AS home_matched_devices,
 SUM(has_confident_home)/NULLIF(COUNT(*),0) AS home_match_rate,
 SUM(IFF(has_confident_home=1,visit_count,0)) AS home_matched_visits,
 COALESCE(COUNT_IF(applied_sample_ratio IS NOT NULL),0) AS weight_eligible_devices,
 SUM(IFF(applied_sample_ratio IS NOT NULL,visit_count,0)) AS weight_eligible_visits,
 SUM(raw_stop_count*applied_sample_ratio) AS adjusted_known_home_raw_stops,
 SUM(visit_count*applied_sample_ratio) AS adjusted_known_home_visits,
 SUM(applied_sample_ratio) AS adjusted_known_home_unique_devices,
 'within_segment_only_not_cross_provider_calibrated' AS panel_comparability
FROM {enriched} GROUP BY 1,2,3,4
), grid AS (
 SELECT c.corridor_id,c.corridor_name,d.year_month,p.column1 AS time_period
 FROM {CORRIDOR_GEO} c CROSS JOIN {final('bay_month_denoms')} d
 CROSS JOIN (VALUES ('all'),('weekdays'),('weekends'),('nine-five'),('evening')) p
)
SELECT g.*,COALESCE(a.raw_stop_count,0) AS raw_stop_count,COALESCE(a.visit_count,0) AS visit_count,
 COALESCE(a.unique_devices,0) AS unique_devices,
 a.raw_stop_count/NULLIF(a.unique_devices,0) AS raw_stops_per_unique_device,
 a.visit_count/NULLIF(a.unique_devices,0) AS visits_per_unique_device,
 COALESCE(a.home_matched_devices,0) AS home_matched_devices,a.home_match_rate,
 COALESCE(a.home_matched_visits,0) AS home_matched_visits,
 COALESCE(a.weight_eligible_devices,0) AS weight_eligible_devices,
 COALESCE(a.weight_eligible_visits,0) AS weight_eligible_visits,
 a.adjusted_known_home_raw_stops,a.adjusted_known_home_visits,a.adjusted_known_home_unique_devices,
 'within_segment_only_not_cross_provider_calibrated' AS panel_comparability
FROM grid g LEFT JOIN a USING(corridor_id,corridor_name,year_month,time_period)""")
publish('corridor_homes_monthly',f"""
SELECT corridor_id,corridor_name,year_month,time_period,block_group_id,home_geography,
 SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count,COUNT(*) AS unique_devices,
 MAX(applied_sample_ratio) AS applied_sample_ratio,MAX(normalization_method) AS normalization_method,
 SUM(raw_stop_count*applied_sample_ratio) AS normalized_raw_stop_count,
 SUM(visit_count*applied_sample_ratio) AS normalized_visit_count,
 SUM(applied_sample_ratio) AS normalized_unique_devices,
 MAX(distance_km) AS approximate_distance_km,
 'within_segment_only_not_cross_provider_calibrated' AS panel_comparability
FROM {enriched} GROUP BY 1,2,3,4,5,6""")
publish('corridor_origin_categories_monthly',f"""
SELECT corridor_id,corridor_name,year_month,time_period,home_geography,
 SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count,COUNT(*) AS unique_devices,
 SUM(raw_stop_count*applied_sample_ratio) AS normalized_raw_stop_count,
 SUM(visit_count*applied_sample_ratio) AS normalized_visit_count,SUM(applied_sample_ratio) AS normalized_unique_devices,
 'within_segment_only_not_cross_provider_calibrated' AS panel_comparability
FROM {enriched} GROUP BY 1,2,3,4,5""")
publish('corridor_home_distance_monthly',f"""
WITH b AS (SELECT *,CASE WHEN distance_km IS NULL THEN 'Unknown distance'
 WHEN distance_km<2 THEN '00: 0–2 km' WHEN distance_km<5 THEN '01: 2–5 km'
 WHEN distance_km<10 THEN '02: 5–10 km' WHEN distance_km<25 THEN '03: 10–25 km'
 WHEN distance_km<50 THEN '04: 25–50 km' WHEN distance_km<100 THEN '05: 50–100 km'
 ELSE '06: 100+ km' END AS distance_band FROM {enriched})
SELECT corridor_id,corridor_name,year_month,time_period,distance_band,
 SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count,COUNT(*) AS unique_devices,
 SUM(raw_stop_count*applied_sample_ratio) AS normalized_raw_stop_count,
 SUM(visit_count*applied_sample_ratio) AS normalized_visit_count,
 SUM(applied_sample_ratio) AS normalized_unique_devices
FROM b GROUP BY 1,2,3,4,5""")
# Half-year aggregation is bounded and respects the six-month ID resets.
for year in range(2019,2026):
    for half in [1,2]:
        mids=[m for m in month_ids if m//100==year and (1 if m%100<=6 else 2)==half]
        label=f'{year}H{half}'
        e='('+union_tables([tab('ENRICHED',m) for m in mids])+')'
        build(P+'REPEAT_'+label,f"""
        WITH devices AS (
         SELECT corridor_id,corridor_name,time_period,cuebiq_id,SUM(raw_stop_count) AS raw_stops,
          SUM(visit_count) AS visit_days,COUNT(*) AS active_months FROM {e} GROUP BY 1,2,3,4)
        SELECT corridor_id,corridor_name,'{label}' AS half_year,time_period,
         SUM(raw_stops) AS raw_stop_count,SUM(visit_days) AS visit_count,COUNT(*) AS unique_devices,
         COALESCE(COUNT_IF(visit_days>=2),0) AS repeat_devices,
         COALESCE(COUNT_IF(visit_days>=2),0)/NULLIF(COUNT(*),0) AS repeat_device_prop,
         COALESCE(COUNT_IF(active_months>=2),0) AS devices_active_multiple_months
        FROM devices GROUP BY 1,2,4""",f'Repeat devices {label}')
        build(P+'HOMES_'+label,f"""
        SELECT corridor_id,corridor_name,'{label}' AS half_year,time_period,block_group_id,home_geography,
         SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count,
         COUNT(DISTINCT cuebiq_id) AS unique_devices_within_half_year,COUNT(*) AS unique_device_months,
         SUM(raw_stop_count*applied_sample_ratio) AS normalized_raw_stop_count,
         SUM(visit_count*applied_sample_ratio) AS normalized_visit_count,
         SUM(applied_sample_ratio) AS normalized_unique_device_months
        FROM {e} GROUP BY 1,2,4,5,6""",f'HOME half-year {label}')
halves=[f'{y}H{h}' for y in range(2019,2026) for h in [1,2]]
publish('corridor_repeat_visitors',union_tables([P+'REPEAT_'+h for h in halves]))
publish('corridor_homes',union_tables([P+'HOMES_'+h for h in halves]))
publish('provider_activity_daily',union_tables([tab('PROVIDER_DAY',m) for m in month_ids]))
publish('provider_home_sample',union_tables([tab('HOME_PROVIDER',m) for m in month_ids]))
publish('extraction_coverage',union_tables([tab('COVERAGE',m) for m in month_ids]))
publish('home_geography',f'SELECT * FROM {HOME_GEO}')
publish('corridor_geography',f'SELECT * FROM {CORRIDOR_GEO}')
publish('corridors_geohash9_diagnostics',f'SELECT *,COUNT(*) OVER(PARTITION BY geohash9) AS matched_corridor_count FROM {EXISTING_COVER}')
publish('run_config',f'SELECT {q(RUN_ID)} AS run_id,{q(json.dumps(config,sort_keys=True))} AS config_json,CURRENT_TIMESTAMP() AS built_at')
publish('home_month_qa',f"""
SELECT year_month,SUM(unique_devices) AS summed_corridor_period_devices,
 SUM(home_matched_devices)/NULLIF(SUM(unique_devices),0) AS weighted_home_match_rate,
 SUM(weight_eligible_devices)/NULLIF(SUM(unique_devices),0) AS weighted_weight_coverage
FROM {final('corridor_stops')} WHERE time_period='all' GROUP BY 1""")


Start: Publish provider_schedule
Submitted: 01c788b1-010c-9cee-0002-4a7603f5acae → DEDICATED.EDDIT_EAST_BAY_70E3F58D7C.provider_schedule
OK: Publish provider_schedule (16.6s observed this session)
Start: Publish provider_coverage_screen
Submitted: 01c788b1-010c-9cee-0002-4a7603f5acbe → DEDICATED.EDDIT_EAST_BAY_70E3F58D7C.provider_coverage_screen
OK: Publish provider_coverage_screen (16.4s observed this session)
Start: Publish bay_month_denoms
Submitted: 01c788b2-010c-9cea-0002-4a7603f57b2a → DEDICATED.EDDIT_EAST_BAY_70E3F58D7C.bay_month_denoms
OK: Publish bay_month_denoms (16.5s observed this session)
Start: Publish home_sample_index
Submitted: 01c788b2-010c-9cee-0002-4a7603f5acd6 → DEDICATED.EDDIT_EAST_BAY_70E3F58D7C.home_sample_index
OK: Publish home_sample_index (16.6s observed this session)
Start: Enrich month 201901
Submitted: 01c788b2-010c-9eb4-0002-4a7603f5c61e → DEDICATED.EDDIT_EAST_BAY_TMP.EBF_70E3F58D7C_ENRICHED_201901
OK: Enrich month 201901 (16.4s observed this session)
Sta

## 8. Final QA and grouped exports

Source completeness is tracked through requested days, months and snapshot counts; days with no matched corridor activity are visible rather than silently interpolated. Raw monthly origin totals must reconcile exactly to corridor totals, including UNKNOWN. Repeat counts cannot exceed unique devices. Bay distance matching failures remain in the Unknown distance bin. Output files contain no device IDs. Large tables are exported in monthly/half-year gzip CSV partitions; smaller tables are single gzip CSVs.

In [9]:
assert len(month_ids)==84
assert int(read(f'SELECT COUNT(*) FROM {final("corridor_stops")}').iloc[0,0])==N_CORRIDORS*84*5
coverage=read(f'SELECT SUM(requested_days) AS requested_days,SUM(days_with_activity) AS days_with_activity FROM {final("extraction_coverage")}')
assert int(coverage.iloc[0].requested_days)==(dt(DATE_END)-dt(DATE_START)).days+1
checks=read(f"""SELECT COUNT(*) AS invalid_rows FROM {final('corridor_stops')}
 WHERE unique_devices>visit_count OR visit_count>raw_stop_count OR home_matched_devices>unique_devices""")
assert int(checks.iloc[0,0])==0
assert int(read(f'SELECT COUNT(*) FROM {final("corridor_repeat_visitors")} WHERE repeat_devices>unique_devices').iloc[0,0])==0
reconcile=read(f"""
WITH h AS (SELECT corridor_id,year_month,time_period,SUM(raw_stop_count) AS stops,SUM(visit_count) AS visits,
 SUM(unique_devices) AS devices FROM {final('corridor_homes_monthly')} GROUP BY 1,2,3)
SELECT COUNT(*) AS differences FROM {final('corridor_stops')} s
FULL JOIN h USING(corridor_id,year_month,time_period)
WHERE s.raw_stop_count IS NULL OR s.raw_stop_count<>COALESCE(h.stops,0) OR s.visit_count<>COALESCE(h.visits,0) OR s.unique_devices<>COALESCE(h.devices,0)""")
assert int(reconcile.iloc[0,0])==0
for name,grain in {
 'corridor_stops':'corridor_id,year_month,time_period',
 'corridor_homes_monthly':'corridor_id,year_month,time_period,block_group_id',
 'home_sample_index':'year_month,norm_key',
 'corridor_repeat_visitors':'corridor_id,half_year,time_period',
 'home_geography':'block_group_id'}.items():
    assert int(read(f'SELECT COUNT(*) FROM (SELECT {grain} FROM {final(name)} GROUP BY {grain} HAVING COUNT(*)>1)').iloc[0,0])==0,name
assert int(read(f'SELECT COUNT(*) FROM {final("bay_month_denoms")}').iloc[0,0])==84
print('Count/grain/reconciliation checks passed.')
display(coverage)
display(read(f'SELECT * FROM {final("home_month_qa")} ORDER BY year_month'))
display(read(f"SELECT normalization_method,COUNT(*) AS cell_months,MIN(applied_sample_ratio) AS min_ratio,MAX(applied_sample_ratio) AS max_ratio FROM {final('home_sample_index')} GROUP BY 1"))
display(read(f"SELECT coordinate_status,COUNT(*) AS cbgs FROM {final('home_geography')} GROUP BY 1"))
if EXPORT_CSV:
    monthly=['corridor_stops','corridor_homes_monthly','corridor_origin_categories_monthly',
             'corridor_home_distance_monthly','home_sample_index','provider_activity_daily']
    halfyear=['corridor_homes','corridor_repeat_visitors']
    small=['provider_schedule','provider_coverage_screen','bay_month_denoms','provider_home_sample','home_geography','corridor_geography',
           'corridors_geohash9_diagnostics','extraction_coverage','home_month_qa','run_config']
    for name in monthly+halfyear+small:
        folder=EXPORT_DIR/name;folder.mkdir(parents=True,exist_ok=True)
        partitions=month_ids if name in monthly else halves if name in halfyear else [None]
        for part in partitions:
            column='year_month' if name in monthly else 'half_year'
            where='' if part is None else f' WHERE {column}={q(part)}'
            out=folder/((str(part) if part is not None else name)+'.csv.gz')
            frame=read('SELECT * FROM '+final(name)+where)
            assert 'cuebiq_id' not in frame.columns
            frame.to_csv(out,index=False,compression='gzip')
        print('Exported',name)
print('Final Snowflake schema:',OUT_SCHEMA)
print('Grouped exports:',EXPORT_DIR.resolve())
print('Interpret adjustment alongside provider mix and HOME coverage. No multi-year unique-person totals are produced.')


Count/grain/reconciliation checks passed.


,requested_days,days_with_activity
0,2557,2557


,year_month,summed_corridor_period_devices,weighted_home_match_rate,weighted_weight_coverage
0,201901,38299,0.785373,0.785373
1,201902,35200,0.832500,0.832500
2,201903,41669,0.813938,0.813938
3,201904,40458,0.819195,0.819195
4,201905,44151,0.809359,0.809359
...,...,...,...,...
79,202508,169927,0.379027,0.379027
80,202509,145778,0.429105,0.429105
81,202510,170872,0.424710,0.424710
82,202511,134948,0.416731,0.416731


,normalization_method,cell_months,min_ratio,max_ratio
0,cbg,271638,0.022399570206,10.705176767677
1,bay_fallback,127866,0.506025016769,1.455558695673


,coordinate_status,cbgs
0,matched,4751


Exported corridor_stops
Exported corridor_homes_monthly
Exported corridor_origin_categories_monthly
Exported corridor_home_distance_monthly
Exported home_sample_index
Exported provider_activity_daily
Exported corridor_homes
Exported corridor_repeat_visitors
Exported provider_schedule
Exported provider_coverage_screen
Exported bay_month_denoms
Exported provider_home_sample
Exported home_geography
Exported corridor_geography
Exported corridors_geohash9_diagnostics
Exported extraction_coverage
Exported home_month_qa
Exported run_config
Final Snowflake schema: DEDICATED.EDDIT_EAST_BAY_70E3F58D7C
Grouped exports: /home/jovyan/eddit-east-bay/activity/east_bay_full/70E3F58D7C
Interpret adjustment alongside provider mix and HOME coverage. No multi-year unique-person totals are produced.
